In [1]:

from xml.etree import ElementTree as ET
from dotenv import load_dotenv
from bs4 import BeautifulSoup
import requests
import json
import os


load_dotenv(dotenv_path=r"D:\Shikha\bias_project\.env")
API_KEY = os.getenv("GOVINFO_API_KEY")

# Verify it loaded
if API_KEY:
    print(f"API key loaded (length: {len(API_KEY)})")
else:
    print("ERROR: API key not found")

BASE_URL = "https://api.govinfo.gov"

print("Setup complete.")
print(f"Base URL: {BASE_URL}")

API key loaded (length: 40)
Setup complete.
Base URL: https://api.govinfo.gov


In [ ]:
# Try a known active session day
# March 12, 2024 was a Tuesday during regular session
package_id = "CREC-2024-03-12"

url = f"{BASE_URL}/packages/{package_id}/summary"
params = {"api_key": API_KEY}

response = requests.get(url, params=params, timeout=30)
print(f"Status: {response.status_code}\n")

if response.status_code == 200:
    summary = response.json()
    
    # Print the key metadata
    print(f"Package ID:    {summary.get('packageId')}")
    print(f"Title:         {summary.get('title')}")
    print(f"Date issued:   {summary.get('dateIssued')}")
    print(f"Doc class:     {summary.get('docClass')}")
    print(f"Pages:         {summary.get('pages')}")
    print(f"Congress:      {summary.get('congress')}")
    print(f"Session:       {summary.get('session')}")
    
    # Show all available download links
    print(f"\nDownload formats available:")
    download = summary.get('download', {})
    for fmt, link in download.items():
        print(f"  {fmt}: {link}")
    
    print(f"\nGranules link: {summary.get('granulesLink')}")
else:
    print(f"Error: {response.text[:500]}")

In [ ]:
# Get the list of granules (each granule = one speech, debate section, or other item)
granules_url = f"{BASE_URL}/packages/{package_id}/granules"
params = {
    "api_key": API_KEY,
    "pageSize": 20,  # just get first 20 for the demo
    "offsetMark": "*"
}

response = requests.get(granules_url, params=params, timeout=30)
print(f"Status: {response.status_code}\n")

if response.status_code == 200:
    data = response.json()
    granules = data.get('granules', [])
    total_count = data.get('count', 0)
    
    print(f"Total granules in this day: {total_count}")
    print(f"Granules returned in this page: {len(granules)}\n")
    
    print("First 10 granules:\n")
    for i, g in enumerate(granules[:10], 1):
        print(f"{i}. {g.get('granuleId')}")
        print(f"   Title:    {g.get('title', '(no title)')[:80]}")
        print(f"   Class:    {g.get('granuleClass')}")
        print()
else:
    print(f"Error: {response.text[:500]}")

In [ ]:
# Pick a granule that looks substantive (not just front matter/procedural)
# We'll look for one with a real title — usually means actual speech content
# Let's grab one with a substantive-sounding title

# First, let's find a good candidate from our granules list
print("Looking for substantive granules (skipping front matter/procedural)...\n")

substantive_granules = []
for g in granules:
    title = g.get('title', '')
    # Skip the obvious procedural items
    if any(skip in title.upper() for skip in [
        'FRONT MATTER', 'DESIGNATION OF SPEAKER', 'PRAYER', 
        'PLEDGE OF ALLEGIANCE', 'JOURNAL', 'ADJOURNMENT',
        'HOUSE OF REPRESENTATIVES'
    ]):
        continue
    substantive_granules.append(g)
    if len(substantive_granules) >= 5:
        break

print(f"Found {len(substantive_granules)} substantive-looking granules:\n")
for i, g in enumerate(substantive_granules, 1):
    print(f"{i}. {g.get('granuleId')}")
    print(f"   Title: {g.get('title', '(no title)')[:100]}")
    print()

# Pick the first one for our demo
if substantive_granules:
    demo_granule = substantive_granules[0]
    demo_granule_id = demo_granule.get('granuleId')
    print(f"\nUsing for demo: {demo_granule_id}")
else:
    print("No substantive granules found in this batch — try fetching more")
    demo_granule_id = granules[3].get('granuleId')  # fallback to 4th item

In [ ]:
# Fetch the granule's full info: summary, metadata XML, and content HTML
granule_url = f"{BASE_URL}/packages/{package_id}/granules/{demo_granule_id}/summary"
params = {"api_key": API_KEY}

response = requests.get(granule_url, params=params, timeout=30)
print(f"Granule summary status: {response.status_code}\n")

if response.status_code == 200:
    granule_summary = response.json()
    
    # Print key fields
    print(f"Granule ID:    {granule_summary.get('granuleId')}")
    print(f"Title:         {granule_summary.get('title')}")
    print(f"Granule class: {granule_summary.get('granuleClass')}")
    print(f"Date:          {granule_summary.get('dateIssued')}")
    
    # Show members associated with this granule (the speakers!)
    members = granule_summary.get('members', [])
    print(f"\nMembers associated with this granule: {len(members)}")
    for m in members:
        print(f"  - {m.get('memberName')}")
        print(f"    State:       {m.get('state')}")
        print(f"    Party:       {m.get('party')}")
        print(f"    Chamber:     {m.get('chamber')}")
        print(f"    BioGuide ID: {m.get('bioGuideId')}")
        print()
    
    # Show available download links for this granule
    print(f"Download formats available:")
    download = granule_summary.get('download', {})
    for fmt, link in download.items():
        print(f"  {fmt}: {link[:80]}...")
else:
    print(f"Error: {response.text[:500]}")

In [ ]:
# Fetch the MODS XML metadata for this granule
mods_link = granule_summary.get('download', {}).get('modsLink')

if mods_link:
    # Append API key
    mods_url = f"{mods_link}?api_key={API_KEY}"
    response = requests.get(mods_url, timeout=30)
    print(f"MODS XML status: {response.status_code}")
    print(f"Content length: {len(response.text)} chars\n")
    
    if response.status_code == 200:
        # Print first 3000 chars to see the structure
        print("First 3000 chars of MODS XML:\n")
        print(response.text[:3000])
else:
    print("No MODS link found")

In [ ]:
# Parse the MODS XML to find congressional members (speakers) with their states
from xml.etree import ElementTree as ET

# The MODS XML uses namespaces — we need to handle them
namespaces = {
    'mods': 'http://www.loc.gov/mods/v3',
}

# Parse the response we already have
mods_xml = response.text
root = ET.fromstring(mods_xml)

# Find all congMember elements (these are the speakers)
# They're typically nested inside <extension> elements
cong_members = root.findall('.//{http://www.loc.gov/mods/v3}extension/congMember')

# Try a few different paths in case the structure varies
if not cong_members:
    # Try without namespace on congMember (it sometimes isn't in the MODS namespace)
    cong_members = root.findall('.//congMember')

if not cong_members:
    # Try finding all elements and searching for congMember-like tags
    all_elems = list(root.iter())
    cong_members = [e for e in all_elems if 'congMember' in e.tag]

print(f"Found {len(cong_members)} congMember elements\n")

for i, member in enumerate(cong_members, 1):
    # Attributes of congMember typically include: bioGuideId, chamber, party, state
    print(f"Speaker {i}:")
    print(f"  BioGuide ID: {member.get('bioGuideId', 'N/A')}")
    print(f"  Chamber:     {member.get('chamber', 'N/A')}")
    print(f"  Party:       {member.get('party', 'N/A')}")
    print(f"  State:       {member.get('state', 'N/A')}")
    print(f"  Type:        {member.get('type', 'N/A')}")
    
    # Get the names (each member has multiple <name> children with different types)
    names = list(member)  # all child elements
    for n in names:
        # The tag may or may not have namespace prefix
        tag = n.tag.split('}')[-1] if '}' in n.tag else n.tag
        if tag == 'name':
            name_type = n.get('type', 'unknown')
            name_value = n.text or ''
            print(f"  Name ({name_type}): {name_value}")
        elif tag == 'role':
            role_value = n.text or ''
            print(f"  Role: {role_value}")
    print()

In [ ]:
# Pick a granule that's clearly a tribute or substantive speech, not a section header
demo_granule_id = "CREC-2024-03-12-pt1-PgH1097-4"  # the "PAYING TRIBUTE..." one

# Fetch its summary
granule_url = f"{BASE_URL}/packages/{package_id}/granules/{demo_granule_id}/summary"
params = {"api_key": API_KEY}

response = requests.get(granule_url, params=params, timeout=30)
print(f"Granule summary status: {response.status_code}\n")

if response.status_code == 200:
    granule_summary = response.json()
    
    print(f"Granule ID:    {granule_summary.get('granuleId')}")
    print(f"Title:         {granule_summary.get('title')}")
    print(f"Granule class: {granule_summary.get('granuleClass')}")
    
    members = granule_summary.get('members', [])
    print(f"\nMembers associated: {len(members)}")
    for m in members:
        print(f"  Name:        {m.get('memberName')}")
        print(f"  State:       {m.get('state')}")
        print(f"  Party:       {m.get('party')}")
        print(f"  Chamber:     {m.get('chamber')}")
        print(f"  BioGuide ID: {m.get('bioGuideId')}")
else:
    print(f"Error: {response.text[:500]}")

In [ ]:
# Now fetch the actual speech text (HTML rendition)
# Try the htm/htmlLink download
download_links = granule_summary.get('download', {})
print("Available download formats:")
for fmt, link in download_links.items():
    print(f"  {fmt}")
print()

# Try to get the txt link (plain text version)
txt_link = download_links.get('txtLink')

if txt_link:
    txt_url = f"{txt_link}?api_key={API_KEY}"
    response = requests.get(txt_url, timeout=30)
    print(f"Text fetch status: {response.status_code}")
    print(f"Content length: {len(response.text)} chars\n")
    
    if response.status_code == 200:
        # Print the speech content
        print("=" * 60)
        print("SPEECH CONTENT:")
        print("=" * 60)
        print(response.text[:3000])  # first 3000 chars
else:
    print("No txt link found. Available links:")
    for fmt, link in download_links.items():
        print(f"  {fmt}: {link[:100]}")

In [ ]:
from bs4 import BeautifulSoup
clean_text = BeautifulSoup(response.text, 'html.parser').get_text()
print(clean_text)